# Lesson 4.3: BERT and Encoder Models

*Module 4 · Large Language Models*

> GPT reads left-to-right. BERT reads in ALL directions at once — like how you actually read. That's why it's incredible at understanding text. Let's learn how it works and build a sentiment analyzer.

---

## About this notebook

This notebook contains the **10 runnable code examples** from the published lesson `lesson-4.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Masked Language Modeling — BERT's Secret Training Trick — `part1_mlm_concept.py`
2. Step 1 — Masked Language Modeling — BERT's Secret Training Trick — `part1_mlm_fun_examples.py`
3. Step 2 — BERT Architecture — "It's Just the Encoder!" — `part2_see_bert_inside.py`
4. Step 3 — Project: Movie Review Sentiment Analyzer — `project_stage_a.py — The 3-line miracle`
5. Step 3 — Project: Movie Review Sentiment Analyzer — `project_stage_b.py — Batch analysis`
6. Step 3 — Project: Movie Review Sentiment Analyzer — `project_stage_c.py — Fine-tune your own model`
7. Step 3 — Project: Movie Review Sentiment Analyzer — `project_stage_d.py — Use your own model!`
8. Step 4 — Transfer Learning — Why Fine-Tuning Works ("NLP's ImageNet Moment") — `transfer_learning.py`
9. Step 5 — The Three Architectures — Encoder vs Decoder vs Encoder-Decoder — `architecture_taxonomy.py`
10. Step 6 — BERT in 2025 — Still Dominant, Surprisingly Relevant — `bert_2025.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch numpy datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Masked Language Modeling — BERT's Secret Training Trick

How do you teach an AI to understand language? Hide a word and make it guess.

**`part1_mlm_concept.py`** · _Block 1 of 10_

MLM: The game BERT plays to learn language — Hide a word, guess it from context.


In [ ]:
# =============================================
# MLM: The game BERT plays to learn language
# Hide a word, guess it from context.
# =============================================

# pip install transformers
from transformers import pipeline

# Load a pre-trained BERT model for fill-in-the-blank
unmasker = pipeline("fill-mask", model="bert-base-uncased")

# Give it a sentence with [MASK] where a word is hidden
sentence = "The cat [MASK] on the warm mat."

print(f"Sentence: {sentence}\n")
print("BERT's top 5 guesses for the hidden word:")

results = unmasker(sentence)
for i, r in enumerate(results):
    print(f"  {i+1}. '{r['token_str']}' ({r['score']:.1%} confident)")

# Output:
#   1. 'sat' (45.2% confident)     ← Correct!
#   2. 'lay' (18.7% confident)     ← Also makes sense
#   3. 'was' (12.3% confident)     ← Grammatically valid
#   4. 'slept' (8.1% confident)    ← Reasonable
#   5. 'landed' (3.4% confident)   ← Less likely but possible


> **What just happened?** BERT looked at ALL the surrounding words ("The", "cat", "on", "the", "warm", "mat") and guessed the hidden word. It read both directions — it used "cat" (before) AND "on the warm mat" (after) to figure out the answer. GPT can only read left-to-right, so it would only see "The cat" and miss all the clues after the blank.


**`part1_mlm_fun_examples.py`** · _Block 2 of 10_

FUN: See what BERT has learned about the world — Just from reading billions of sentences!


In [ ]:
# =============================================
# FUN: See what BERT has learned about the world
# Just from reading billions of sentences!
# =============================================

from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

test_sentences = [
    # Does it know geography?
    "The capital of France is [MASK].",
    
    # Does it understand professions?
    "The doctor told the [MASK] to take rest.",
    
    # Does it understand food?
    "I ordered a plate of butter [MASK] for dinner.",
    
    # Does it understand emotions?
    "After winning the match, the team felt very [MASK].",
    
    # Does it understand code/tech?
    "Python is a popular programming [MASK].",
]

for sent in test_sentences:
    result = unmasker(sent)[0]  # Top 1 guess
    print(f"  {sent}")
    print(f"  → BERT guesses: '{result['token_str']}' ({result['score']:.0%})\n")

# BERT learned geography, professions, food, emotions, and tech
# ALL from just playing fill-in-the-blank billions of times!
# Nobody told it "Paris is the capital of France."
# It figured it out from the patterns in text.


### Step 2 · BERT Architecture — "It's Just the Encoder!"

Good news: you already built most of BERT in Lesson 4.4.

**`part2_see_bert_inside.py`** · _Block 3 of 10_

PEEK INSIDE BERT — See how it processes a sentence into vectors


In [ ]:
# =============================================
# PEEK INSIDE BERT
# See how it processes a sentence into vectors
# =============================================

from transformers import BertTokenizer, BertModel
import torch

# Load BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

# Tokenize a sentence
sentence = "I love this movie"
tokens = tokenizer(sentence, return_tensors="pt")

# See what tokens BERT created
token_ids = tokens["input_ids"][0]
token_words = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens BERT sees:")
for i, (tid, word) in enumerate(zip(token_ids, token_words)):
    print(f"  Position {i}: '{word}' (ID: {tid})")

# Output:
#   Position 0: '[CLS]'  (ID: 101)    ← Special start token
#   Position 1: 'i'      (ID: 1045)
#   Position 2: 'love'   (ID: 2293)
#   Position 3: 'this'   (ID: 2023)
#   Position 4: 'movie'  (ID: 3185)
#   Position 5: '[SEP]'  (ID: 102)    ← Special end token

# Pass through BERT
with torch.no_grad():
    outputs = model(**tokens)

# BERT gives us a vector for EACH token
all_vectors = outputs.last_hidden_state
print(f"\nOutput shape: {all_vectors.shape}")
# → (1, 6, 768)
# 1 sentence, 6 tokens, 768 numbers per token

# The [CLS] vector = a summary of the WHOLE sentence
cls_vector = all_vectors[0][0]
print(f"[CLS] vector (sentence summary): {cls_vector.shape}")
# → 768 numbers that capture the meaning of "I love this movie"
# We'll use this for sentiment analysis in Part 3!


> **What just happened?** 🧠 Knowledge Check What is the [CLS] token and how does BERT use it for classification? The [CLS] token is the class label that BERT predicts The [CLS] token's 768-dim output vector summarizes the whole sentence; a linear layer on top converts it to class probabilities The [CLS] token marks where the sentence ends Check Answer BERT took our sentence "I love this movie", added [CLS] at the start and [SEP] at the end, processed it through 12 layers of attention + feed-forward, and gave us a 768-number vector for each token. The [CLS] vector is special — it's a compressed summary of the entire sentence's meaning. We'll use it to classify sentiment next!


### Step 3 · Project: Movie Review Sentiment Analyzer

Let's build something real using BERT and Hugging Face.

#### Stage A: The 3-Line Version (instant gratification)

**`project_stage_a.py — The 3-line miracle`** · _Block 4 of 10_

STAGE A: Sentiment analysis in 3 lines — This uses a BERT model already fine-tuned


In [ ]:
# =============================================
# STAGE A: Sentiment analysis in 3 lines
# This uses a BERT model already fine-tuned
# for sentiment analysis. No training needed!
# =============================================

from transformers import pipeline

# Load a pre-trained sentiment model (BERT inside)
sentiment = pipeline("sentiment-analysis")

# Analyze!
result = sentiment("I absolutely loved this movie! Best film of the year.")
print(result)
# → [{'label': 'POSITIVE', 'score': 0.9998}]

# That's it. 3 lines. BERT is doing all the heavy lifting inside.


> **What just happened?** Hugging Face downloaded a pre-trained BERT model that someone already trained on thousands of movie reviews. We just used it. Three lines of code. It works because someone else already did the training — we'll learn how they did it in Stage C.


#### Stage B: Analyze Multiple Reviews

**`project_stage_b.py — Batch analysis`** · _Block 5 of 10_

STAGE B: Analyze a bunch of reviews at once


In [ ]:
# =============================================
# STAGE B: Analyze a bunch of reviews at once
# =============================================

from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie I've ever seen. Complete waste of time.",
    "It was okay. Nothing special but not terrible either.",
    "The visuals were stunning but the story was weak.",
    "I walked out of the theater crying. So beautiful.",
    "Boring. I fell asleep halfway through.",
    "A masterpiece. Will watch it again and again!",
]

print("Movie Review Sentiment Analysis")
print("=" * 55)

for review in reviews:
    result = sentiment(review)[0]
    label = result["label"]
    score = result["score"]
    
    # Color-code the output
    emoji = "💚" if label == "POSITIVE" else "🔴"
    
    print(f"\n{emoji} {label} ({score:.0%})")
    print(f"   \"{review[:60]}...\"")

# =============================================
# Count positive vs negative
# =============================================
results = sentiment(reviews)
pos = sum(1 for r in results if r["label"] == "POSITIVE")
neg = len(results) - pos

print(f"\n{'=' * 55}")
print(f"Summary: {pos} positive, {neg} negative out of {len(reviews)} reviews")


#### Stage C: Fine-Tune BERT on Your Own Data

**`project_stage_c.py — Fine-tune your own model`** · _Block 6 of 10_

STAGE C: Fine-tune BERT for sentiment analysis — We'll use a small subset of IMDB reviews.


In [ ]:
# =============================================
# STAGE C: Fine-tune BERT for sentiment analysis
# We'll use a small subset of IMDB reviews.
# =============================================

# pip install transformers datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import numpy as np

# ── Step 1: Load the data ──
print("Loading IMDB movie reviews...")
dataset = load_dataset("imdb")

# Use a small subset for fast training (1000 train, 200 test)
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_test  = dataset["test"].shuffle(seed=42).select(range(200))

# Peek at the data
print(f"\nExample review (first 100 chars):")
print(f"  Text:  '{small_train[0]['text'][:100]}...'")
print(f"  Label: {small_train[0]['label']} (0=negative, 1=positive)")

# ── Step 2: Prepare the data for BERT ──
print("\nTokenizing reviews for BERT...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,      # cut long reviews to 512 tokens
        padding="max_length",  # pad short reviews to same length
        max_length=256,        # keep it manageable
    )

train_data = small_train.map(tokenize, batched=True)
test_data  = small_test.map(tokenize, batched=True)

# ── Step 3: Load BERT with a classification head ──
print("Loading DistilBERT + adding classification head...")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,   # 2 classes: positive and negative
)

# ── Step 4: Set up training ──
training_args = TrainingArguments(
    output_dir="./my_sentiment_model",
    num_train_epochs=3,            # 3 passes through the data
    per_device_train_batch_size=16, # 16 reviews at a time
    per_device_eval_batch_size=16,
    eval_strategy="epoch",    # test after each pass
    logging_steps=50,
    save_strategy="no",             # don't save checkpoints (for speed)
)

# Accuracy calculator
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    accuracy = (predictions == eval_pred.label_ids).mean()
    return {"accuracy": accuracy}

# ── Step 5: TRAIN! ──
print("\nTraining started!\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

# ── Step 6: See the results! ──
results = trainer.evaluate()
print(f"\nFinal accuracy: {results['eval_accuracy']:.1%}")
print("(With just 1000 training examples! More data = better accuracy)")


> **What just happened?** 🧠 Knowledge Check Why is BERT 250x cheaper than GPT-4o for classification at scale? BERT has fewer parameters so it runs faster BERT processes the entire input in a single forward pass, while GPT-4o must generate output tokens one-by-one via API calls GPT-4o is intentionally overpriced by OpenAI Check Answer We took a pre-trained DistilBERT (a faster version of BERT), added a tiny classification layer on top (just 2 outputs: positive/negative), and trained it on 1000 IMDB movie reviews. In just 3 epochs (~2 minutes on a GPU), it learned to classify sentiment with ~85-90% accuracy. That's the magic of fine-tuning — BERT already understands English, it just needed a few examples to learn this specific task .


#### Stage D: Use Your Trained Model

**`project_stage_d.py — Use your own model!`** · _Block 7 of 10_

STAGE D: Use your fine-tuned model


In [ ]:
# =============================================
# STAGE D: Use your fine-tuned model
# =============================================

from transformers import pipeline

# Load YOUR trained model (not someone else's!)
my_model = pipeline(
    "sentiment-analysis",
    model=model,          # the model we just trained
    tokenizer=tokenizer,
)

# Test with reviews the model has NEVER seen
test_reviews = [
    "The acting was phenomenal. I was on the edge of my seat!",
    "I want my 2 hours back. Terrible plot, terrible acting.",
    "Not bad, not great. A decent one-time watch.",
    "My kids loved it and honestly, so did I!",
    "The trailer was better than the actual movie.",
]

print("Your custom sentiment model says:\n")
for review in test_reviews:
    result = my_model(review)[0]
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    score = result["score"]
    emoji = "👍" if label == "POSITIVE" else "👎"
    
    print(f"  {emoji} {label} ({score:.0%})")
    print(f"     \"{review[:55]}...\"\n")


> **What just happened?** You just used YOUR OWN trained model to analyze brand-new reviews it has never seen before. You went from zero to a working sentiment analyzer in 4 stages: (A) use someone else's model in 3 lines, (B) analyze multiple reviews, (C) train your own model on real data, (D) use your trained model in production.


### Step 4 · Transfer Learning — Why Fine-Tuning Works ("NLP's ImageNet Moment")

The most important concept behind BERT: pre-train once on billions of words, then fine-tune on YOUR task with minimal data.

**`transfer_learning.py`** · _Block 8 of 10_

TRANSFER LEARNING — The two-stage paradigm


In [ ]:
# =============================================
# TRANSFER LEARNING — The two-stage paradigm
# =============================================

# Stage 1: PRE-TRAINING (Google did this for you)
#   - Trained on Wikipedia + BookCorpus (3.3B words)
#   - 64 TPUs × 4 days = ~$50,000+ compute
#   - Learned: grammar, facts, reasoning, world knowledge
#   - Output: bert-base-uncased (110M parameters)

# Stage 2: FINE-TUNING (you do this in 15 minutes)
#   - Take pre-trained BERT
#   - Add a tiny classification head (768 → 2 neurons)
#   - Train on YOUR data (even 200 samples works!)
#   - Result: 90%+ accuracy on sentiment, NER, QA, etc.

# This is like how ImageNet pre-training revolutionized
# computer vision — Sebastian Ruder called it "NLP's ImageNet moment"

# The [CLS] token: how a language model becomes a classifier
# Input:  [CLS] This movie was amazing [SEP]
# BERT:   Produces 768-dim vector for each token
# [CLS] vector → aggregates the whole sentence's meaning
# Classification head: [CLS] vector → Linear(768, 2) → positive/negative

from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "This movie was absolutely fantastic!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    cls_vector = outputs.last_hidden_state[:, 0, :]  # [CLS] = position 0

print(f"[CLS] vector shape: {cls_vector.shape}")  # [1, 768]
print(f"This 768-dim vector IS the sentence's meaning.")
print(f"Add Linear(768, 2) on top → instant sentiment classifier.")

> **What just happened?** 🧠 Knowledge Check What is ModernBERT and why was it released in December 2024? ModernBERT is OpenAI's replacement for GPT-4 ModernBERT updates BERT with modern techniques: 8K context, RoPE, FlashAttention, trained on 2T tokens including code ModernBERT is just BERT with more training data Check Answer Transfer learning is the key insight: Google spent $50K+ pre-training BERT on 3.3B words. You spend 15 minutes fine-tuning on 200 samples. The [CLS] token's 768-dimensional vector summarizes the entire sentence's meaning — add a simple linear layer on top and you have a classifier. This is why BERT was revolutionary: it made NLP accessible to anyone with a small labeled dataset.


### Step 5 · The Three Architectures — Encoder vs Decoder vs Encoder-Decoder

The Transformer from Lesson 4.4 can be split three ways. Each has different superpowers.

**`architecture_taxonomy.py`** · _Block 9 of 10_

THE THREE TRANSFORMER ARCHITECTURES


In [ ]:
# =============================================
# THE THREE TRANSFORMER ARCHITECTURES
# =============================================
from transformers import pipeline

# 1. ENCODER-ONLY (BERT) — understands text
#    Attention: bidirectional (every token sees every other)
#    Tasks: classification, NER, extractive QA, embeddings
#    Models: BERT, RoBERTa, DeBERTa, ModernBERT, IndicBERT
classifier = pipeline("sentiment-analysis")
print(classifier("BERT understands text deeply"))

# 2. DECODER-ONLY (GPT) — generates text
#    Attention: causal (each token only sees past tokens)
#    Tasks: chat, code gen, creative writing, reasoning
#    Models: GPT-4, Claude, Gemini, LLaMA, DeepSeek
generator = pipeline("text-generation", model="gpt2")
print(generator("The future of AI is", max_length=20))

# 3. ENCODER-DECODER (T5) — transforms text
#    Attention: encoder=bidirectional, decoder=causal + cross-attention
#    Tasks: translation, summarization, seq-to-seq
#    Models: T5, BART, mBART, Flan-T5
summarizer = pipeline("summarization", model="t5-small")

# The "bank" example — why bidirectional matters:
# "I went to the bank to deposit money"
#   GPT encoding "bank": only sees "I went to the" → ambiguous!
#   BERT encoding "bank": sees "deposit money" too → financial bank!
# This is why BERT is better at UNDERSTANDING tasks.

# From Lesson 4.1: Word2Vec gave "bank" ONE vector (static)
# BERT gives "bank" DIFFERENT vectors depending on context
# → "river bank" ≠ "bank account" → contextual embeddings!


> **What just happened?** Three architectures, three superpowers. Encoder (BERT): bidirectional attention → best for understanding. Decoder (GPT): causal attention → best for generating. Encoder-Decoder (T5): both → best for transforming (translation, summarization). The "bank" example shows why bidirectional context matters: BERT sees the whole sentence and knows it's a financial bank, while GPT reading left-to-right hasn't seen "deposit" yet. Connection to Lesson 4.1: Word2Vec gave "bank" one static vector. BERT gives it different vectors in different contexts — this is the contextual embedding breakthrough.


### Step 6 · BERT in 2025 — Still Dominant, Surprisingly Relevant

Encoder models get 3× more HuggingFace downloads than decoders. BERT itself: 68M+ downloads/month.

**`bert_2025.py`** · _Block 10 of 10_

BERT IN 2025 — Variants, Cost, Production


In [ ]:
# =============================================
# BERT IN 2025 — Variants, Cost, Production
# =============================================

# BERT Family Tree — what to use when
variants = {
    "BERT-base":     (110, 512,   "The original. Still works great."),
    "DistilBERT":    (66,  512,   "40% smaller, 60% faster, 95% accuracy"),
    "RoBERTa":       (125, 512,   "No NSP, more data → better than BERT"),
    "DeBERTa-v3":    (184, 512,   "Disentangled attention → best accuracy"),
    "ModernBERT":    (149, 8192,  "Dec 2024: 16x context, 2T tokens, code!"),
    "IndicBERT":     (33,  512,   "12 Indian languages (AI4Bharat/IIT-M)"),
}

print(f"{'Model':<16} {'Params':>7} {'Context':>8} Description")
print("-" * 70)
for name, (params, ctx, desc) in variants.items():
    print(f"  {name:<14} {params:>5}M {ctx:>7} {desc}")

# THE COST ARGUMENT — why BERT wins in production
print(f"""
COST: Classifying 1 million texts
  Fine-tuned BERT:     $2-5    (GPU compute)
  GPT-4o API:          $1,250  (token pricing)
  GPT-4o-mini API:     $67     (still 13x more expensive)

  BERT is 250-500x cheaper than GPT-4o for classification!
  And with 200 labeled samples, BERT matches GPT-4o accuracy.

ModernBERT (Dec 2024) — "What BERT would look like if built today":
  - 8,192 token context (vs BERT's 512)
  - Trained on 2 TRILLION tokens (vs BERT's 3.3B)
  - RoPE + FlashAttention-2 (from Lesson 4.4!)
  - First encoder model that understands CODE
  - Apache 2.0 license, 2x faster than DeBERTa

For Indian languages: IndicBERT covers Hindi, Telugu, Tamil, 
  Kannada, Malayalam, Bengali, Marathi, Gujarati + 4 more.
  Built by AI4Bharat (IIT Madras) on 9B tokens from IndicCorp.

BERT in RAG (Module 8):
  - Bi-encoder (Sentence-BERT): generates embeddings for retrieval
  - Cross-encoder: reranks top candidates for precision
  - Bi-encoder retrieves top 50 → Cross-encoder reranks to top 5 → LLM answers
""")


> **What just happened?** BERT isn't dead — it's the most deployed model family in production NLP. 68M+ monthly downloads , 3x more than decoders. The cost advantage is massive: $2-5 vs $1,250 for classifying 1M texts. ModernBERT (Dec 2024) modernizes the architecture with 8K context, RoPE, and FlashAttention. IndicBERT covers 12 Indian languages. In RAG pipelines (Module 8), BERT-based models power both retrieval (bi-encoders) and reranking (cross-encoders).


---

## Wrap-up

You've walked through all 10 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
